# 02 – Sample Definition
Defines the modeling population, target variable, and out-of-time train/test split.

## Which vintages have matured?
The data snapshot is from 2020. Loans still outstanding at that point have unknown
outcomes and must be excluded, but excluding them biases the sample toward early
defaulters and early payoffs. To avoid that, I only keep vintages where nearly all
loans had resolved.

In [4]:
import pandas as pd
df = pd.read_parquet("../data/loans_raw.parquet")

In [5]:
df["term"].unique()

<ArrowStringArray>
[' 36 months', ' 60 months']
Length: 2, dtype: str

In [6]:
mask = (df["term"] == " 36 months") & (df["issue_d"].dt.year.between(2013, 2017)) & (df["application_type"] == "Individual")
df36 = df[mask].copy()

In [7]:
df36.shape

(1160687, 141)

In [8]:
df["loan_status"].unique()

<ArrowStringArray>
[                                         'Fully Paid',
                                         'Charged Off',
  'Does not meet the credit policy. Status:Fully Paid',
 'Does not meet the credit policy. Status:Charged Off',
                                             'Current',
                                  'Late (31-120 days)',
                                     'In Grace Period',
                                   'Late (16-30 days)',
                                             'Default',
                                              'Issued']
Length: 10, dtype: str

In [9]:
resolved_statuses = ["Fully Paid","Charged Off","Default"]
df36["unresolved"] = ~df36["loan_status"].isin(resolved_statuses)

In [10]:
df36.shape

(1160687, 142)

In [11]:
df36["quarter"] = df36["issue_d"].dt.to_period('Q')

In [12]:
df36.groupby("quarter")["unresolved"].agg(["mean", "count"])

,mean,count
quarter,,
2013Q1,0.000000,17988
2013Q2,0.000000,23136
2013Q3,0.000000,27318
2013Q4,0.000000,31980
2014Q1,0.000000,34074
2014Q2,0.000000,37881
2014Q3,0.000000,40595
2014Q4,0.000000,50020
2015Q1,0.000000,56569


**Finding:** Share of unresolved loans is ~0% through 2016Q2, 0.6% in 2017Q1, then
jumps to 14% in 2017Q2 and 40% by 2017Q4.

**Decision:** Test window ends at 2017Q1. Train = 2013–2015, Test = 2016Q1–2017Q1.

In [13]:
mask = (df36["quarter"] <= "2017Q1") & (~df36["unresolved"])
model_df = df36[mask].copy()

## Target definition
- **Bad (1):** Charged Off, Default (121+ days past due, near-certain charge-off)
- **Good (0):** Fully Paid
- **Excluded:** Current, Late, In Grace Period, Issued — outcome not yet known

In [14]:
model_df["default"] = model_df["loan_status"].isin(["Charged Off","Default"]).astype(int)

In [15]:
model_df.shape

(933498, 144)

In [16]:
model_df["default"].mean()

np.float64(0.1471401117088628)

In [17]:
model_df["year"] = model_df["issue_d"].dt.year
model_df.groupby("year")["default"].agg(["mean", "count"])

,mean,count
year,,
2013,0.123260,100422
2014,0.137264,162570
2015,0.148922,282934
2016,0.158848,318223
2017,0.143881,69349


**Finding:** Bad rate is ~14.0% in train vs ~15.6% in test, rising from 2013–2016.
Expect the model to under-predict PD on the test set; check calibration in modeling.

In [18]:
import numpy as np
model_df["sample"] = np.where(model_df["year"] < 2016, "train", "test")

In [19]:
model_df.to_parquet("../data/model_sample.parquet", index=False)